In [1]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 58.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import pdfplumber
import pandas as pd
import os
import glob
import re  # ไลบรารีสำคัญสำหรับค้นหา Pattern ของข้อความ
# เชื่อมต่อกับ Google Drive เพื่อเข้าถึงโฟลเดอร์ที่เก็บไฟล์ข้อมูลดิบ
drive.mount('/content/drive')
# กำหนดโฟลเดอร์และสร้าง Dictionary ดักจับ OCR Error
# ปัญหา: ข้อมูลกรมอุตุฯ แบ่งเป็น 2 ฟอร์แมตหลัก (ปี 63-64 แบบนึง, ปี 65-67 อีกแบบนึง)
folder_63_64 = '/content/drive/MyDrive/Data_Sci_Pro/Data_Set/Rain/Rain_63_64'
folder_65_67 = '/content/drive/MyDrive/Data_Sci_Pro/Data_Set/Rain/Rain_65_67'

# ดักจับชื่อเดือนที่มีปัญหาการสะกด
thai_months = {
    'มกรา': 'Jan', 'มกราคม': 'Jan', 'กุมภา': 'Feb', 'กมภาพนธ': 'Feb',
    'มีนา': 'Mar', 'มนาคม': 'Mar', 'เมษา': 'Apr', 'พฤษภา': 'May',
    'มิถุนา': 'Jun', 'มถนายน': 'Jun', 'กรกฎา': 'Jul', 'สิงหา': 'Aug', 'สงหาคม': 'Aug',
    'กันยา': 'Sep', 'กนยายน': 'Sep', 'ตุลา': 'Oct', 'ตลาคม': 'Oct',
    'พฤศจิกา': 'Nov', 'พฤศจกายน': 'Nov', 'ธันวา': 'Dec', 'ธนวาคม': 'Dec'
}

def extract_month_year(filename):
    year_match = re.search(r'256[3-7]|6[3-7]', filename)
    year = "20" + str(int(year_match.group()[-2:]) - 43) if year_match else "Unknown"
    month_eng = "Unknown"
    for th, en in thai_months.items():
        if th in filename: month_eng = en; break
    return month_eng, year

def parse_rain_text(folder_path, format_type):
    pdf_files = glob.glob(os.path.join(folder_path, '*.pdf'))
    results = []
    total_files = len(pdf_files)

    print(f"\n เริ่มต้นอ่านข้อมูลใน {format_type} (จำนวนทั้งหมด {total_files} ไฟล์)")

    # เพิ่ม enumerate เพื่อใช้นับลำดับไฟล์ (index)
    for index, file in enumerate(pdf_files, start=1):
        filename = os.path.basename(file)

        # [จุดที่เพิ่มเข้ามา] แสดงสถานะว่ากำลังทำงานกับไฟล์ไหน
        print(f"  ⏳ [{index}/{total_files}] กำลังแกะข้อมูลไฟล์: {filename} ...")

        month, year = extract_month_year(filename)
        if month == "Unknown" or year == "Unknown":
            print(f"     ข้ามไฟล์ {filename} (ไม่สามารถระบุเดือน/ปีจากชื่อไฟล์ได้)")
            continue

        with pdfplumber.open(file) as pdf:
            for page in pdf.pages:
                lines = page.extract_text().split('\n') if page.extract_text() else []
                for line in lines:
                    match = re.match(r'^([ก-๙a-zA-Z]{3,}(?:\s[ก-๙a-zA-Z]{2,})?)\s+(-?\d+\.\d+|-|T)\s+(-?\d+\.\d+|-|T)', line)
                    if match:
                        station = match.group(1).strip()
                        if station in ['ภาค', 'สถานี', 'Mean', 'Actual']: continue

                        numbers = re.findall(r'-?\d+\.\d+|-|T', line)
                        try:
                            if format_type == 'Old_Format' and len(numbers) >= 1:
                                rain_val = 0.0 if numbers[0] in ['-', 'T'] else float(numbers[0])
                            elif format_type == 'New_Format' and len(numbers) >= 3:
                                rain_val = 0.0 if numbers[2] in ['-', 'T'] else float(numbers[2])
                            else:
                                continue

                            results.append({'Year': year, 'Month': month, 'Province': station, 'Rainfall': rain_val})
                        except: pass
    return results

# เริ่มรันการสกัดข้อมูล และประกอบร่างตาราง
data_old = parse_rain_text(folder_63_64, 'Old_Format')
data_new = parse_rain_text(folder_65_67, 'New_Format')
df_rain = pd.DataFrame(data_old + data_new)

if df_rain.empty:
    print("\nไม่พบข้อมูลฝนในไฟล์ PDF ที่ระบุ โปรดตรวจสอบเส้นทางไฟล์")
else:
    # ทำความสะอาดชื่อจังหวัด (Data Standardization)
    # ปัญหา: กรมอุตุฯ เขียนชื่อจังหวัดสลับไปมา ทั้งภาษาอังกฤษ ไทย และคำที่สะกดผิด
    df_rain['Province'] = df_rain['Province'].replace({
        'CHIANG MAI': 'เชียงใหม่', 'PHUKET': 'ภูเก็ต', 'ลา ปาง': 'ลำปาง', 'ขอนแกน': 'ขอนแก่น',
        'BANGKOK': 'กรุงเทพมหานคร', 'CHONBURI': 'ชลบุรี', 'RAYONG': 'ระยอง',
        'ลาพูน': 'ลำพูน', 'กาแพงเพชร': 'กำแพงเพชร', 'แมฮองสอน': 'แม่ฮ่องสอน',
        'สระบรี': 'สระบุรี', 'อทยธานี': 'อุทัยธานี', 'พัทลง': 'พัทลุง', 'เพชรบร': 'เพชรบุรี'
    })

    # ยุบรวมสถานี : หากจังหวัดไหนมีจุดวัดฝนหลายแห่ง ให้หาค่าเฉลี่ย
    df_rain_clean = df_rain.groupby(['Year', 'Month', 'Province'], as_index=False)['Rainfall'].mean()

    # กระบวนการเติมค่าว่าง
    # ปัญหา: บางเดือน บางจังหวัด เซนเซอร์อาจเสีย ทำให้ไม่มี Record บันทึกไว้
    # วิธีแก้: เติมค่าว่างด้วย "ค่าเฉลี่ยฝนของเดือนนั้นๆ"
    df_rain_imputed = df_rain_clean.copy()
    df_rain_imputed['Rainfall'] = df_rain_imputed.groupby(['Year', 'Month'])['Rainfall'].transform(lambda x: x.fillna(x.mean()))

    df_rain_imputed.to_csv('Cleaned_Rainfall_Monthly.csv', index=False, encoding='utf-8-sig')
    print(f"ข้อมูลทั้งหมด {len(df_rain_imputed)} แถว")
    display(df_rain_imputed.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

 เริ่มต้นอ่านข้อมูลใน Old_Format (จำนวนทั้งหมด 24 ไฟล์)
  ⏳ [1/24] กำลังแกะข้อมูลไฟล์: ข้อมูลสรุปลักษณะอากาศรายเดือนมกราคม 2563.pdf ...
  ⏳ [2/24] กำลังแกะข้อมูลไฟล์: ข้อมูลสรุปลักษณะอากาศรายเดือนกุมภาพันธ์ 2563.pdf ...
  ⏳ [3/24] กำลังแกะข้อมูลไฟล์: รายงานอุตุนิยมวิทยาเกษตรเดือนมีนาคม 2563.pdf ...
  ⏳ [4/24] กำลังแกะข้อมูลไฟล์: รายงานอุตุนิยมวิทยาเกษตรเดือนเมษายน 2563.pdf ...
  ⏳ [5/24] กำลังแกะข้อมูลไฟล์: รายงานอุตุนิยมวิทยาเกษตรเดือนพฤษภาคม 2563.pdf ...
  ⏳ [6/24] กำลังแกะข้อมูลไฟล์: รายงานอุตุนิยมวิทยาเกษตรเดือนมิถุนายน 2563.pdf ...
  ⏳ [7/24] กำลังแกะข้อมูลไฟล์: รายงานอุตุนิยมวิทยาเกษตรเดือนกรกฎาคม 2563.pdf ...
  ⏳ [8/24] กำลังแกะข้อมูลไฟล์: รายงานอุตุนิยมวิทยาเกษตรเดือนสิงหาคม 2563.pdf ...
  ⏳ [9/24] กำลังแกะข้อมูลไฟล์: รายงานอุตุนิยมวิทยาเกษตรเดือนกันยายน 2563.pdf ...
  ⏳ [10/24] กำลังแกะข้อมูลไฟล์: รายงานอุตุนิยมวิทยาเกษตรเดือนตุลาคม 2563.pdf ...
  ⏳

,Year,Month,Province,Rainfall
0,2022,Apr,Aranyaprathet,96.35
1,2022,Apr,Bangkok Airport,145.35
2,2022,Apr,Bangkok Metropolis,125.90
3,2022,Apr,Bhumibol Dam,24.35
4,2022,Apr,Bua Chum,94.25


In [ ]:
import pdfplumber
import pandas as pd
import os
import glob
import re
import warnings
warnings.filterwarnings('ignore')

# กำหนดเส้นทางไฟล์: โฟลเดอร์เก็บ PDF ปี 63-64 และไฟล์ CSV ปี 65-67 ที่คลีนไว้แล้ว
folder_63_64 = '/content/drive/MyDrive/Data_Sci_Pro/Data_Set/Rain/Rain_63_64'
# แก้ไขพาธของไฟล์ CSV เพื่อให้ชี้ไปยังตำแหน่งที่ถูกต้องที่บันทึกไว้ในเซลล์ก่อนหน้า
path_cleaned_65_67 = '/content/Cleaned_Rainfall_Monthly.csv'

# การสร้าง Dictionary เพื่อทำ Data Standardization
# Dictionary 1: แปลงชื่อเดือนภาษาไทยที่ดึงจากชื่อไฟล์ ให้เป็นชื่อย่อภาษาอังกฤษมาตรฐาน
thai_months = {
    'มกราคม': 'Jan', 'กุมภาพันธ์': 'Feb', 'มีนาคม': 'Mar', 'เมษายน': 'Apr',
    'พฤษภาคม': 'May', 'มิถุนายน': 'Jun', 'กรกฎาคม': 'Jul', 'สิงหาคม': 'Aug',
    'กันยายน': 'Sep', 'ตุลาคม': 'Oct', 'พฤศจิกายน': 'Nov', 'ธันวาคม': 'Dec'
}

# Dictionary 2: ปัญหาใหญ่ของปี 63-64 คือชื่อสถานีเป็น "ภาษาไทย" และบางสถานีไม่ใช่ชื่อจังหวัด
# เช่น สถานี 'ดอยมูเซอ' ต้อง Map ให้เป็นจังหวัด 'ตาก', 'บางนา' ต้องเป็น 'กรุงเทพมหานคร'
thai_station_to_prov = {
    'เชียงราย': 'เชียงราย', 'เชียงใหม่': 'เชียงใหม่', 'ดอยมูเซอ': 'ตาก', 'ตากฟ้า': 'นครสวรรค์',
    'ลำปาง': 'ลำปาง', 'น่าน': 'น่าน', 'ศรีสำโรง': 'สุโขทัย', 'พิจิตร': 'พิจิตร',
    'เลย': 'เลย', 'สกลนคร': 'สกลนคร', 'นครพนม': 'นครพนม', 'ท่าพระ': 'ขอนแก่น',
    'ร้อยเอ็ด': 'ร้อยเอ็ด', 'อุบลราชธานี': 'อุบลราชธานี', 'ศรีสะเกษ': 'ศรีสะเกษ',
    'ปากช่อง': 'นครราชสีมา', 'สุรินทร์': 'สุรินทร์', 'ชัยนาท': 'ชัยนาท',
    'อยุธยา': 'พระนครศรีอยุธยา', 'ปทุมธานี': 'ปทุมธานี', 'ราชบุรี': 'ราชบุรี',
    'อู่ทอง': 'สุพรรณบุรี', 'กำแพงแสน': 'นครปฐม', 'บางนา': 'กรุงเทพมหานคร',
    'ฉะเชิงเทรา': 'ฉะเชิงเทรา', 'ห้วยโป่ง': 'ระยอง', 'พลิ้ว': 'จันทบุรี',
    'หนองพลับ': 'ประจวบคีรีขันธ์', 'สวี': 'ชุมพร', 'สุราษฎร์ธานี': 'สุราษฎร์ธานี',
    'นครศรีธรรมราช': 'นครศรีธรรมราช', 'พัทลุง': 'พัทลุง', 'คอหงษ์': 'สงขลา', 'ยะลา': 'ยะลา'
}

# Dictionary 3: ปัญหาใหญ่ของปี 65-67 คือชื่อสถานีเปลี่ยนไปใช้ "ภาษาอังกฤษ"
# จึงต้องสร้าง Dictionary เพื่อ Map กลับมาเป็นชื่อจังหวัด "ภาษาไทย" ให้ตรงกับปี 63-64
eng_station_to_prov = {
    'Bangkok Metropolis': 'กรุงเทพมหานคร', 'Bangkok Airport': 'กรุงเทพมหานคร',
    'Chiang Mai': 'เชียงใหม่', 'Chiang Rai': 'เชียงราย', 'Chon Buri': 'ชลบุรี',
    'Chumphon': 'ชุมพร', 'Khon Kaen': 'ขอนแก่น', 'Lampang': 'ลำปาง', 'Lamphun': 'ลำพูน',
    'Loei': 'เลย', 'Nakhon Phanom': 'นครพนม', 'Nakhon Ratchasima': 'นครราชสีมา',
    'Nakhon Sawan': 'นครสวรรค์', 'Nakhon Si Thammarat': 'นครศรีธรรมราช',
    'Narathiwat': 'นราธิวาส', 'Nong Khai': 'หนองคาย', 'Pathum Thani': 'ปทุมธานี',
    'Pattani': 'ปัตตานี', 'Phangnga': 'พังงา', 'Phatthalung': 'พัทลุง',
    'Phayao': 'พะเยา', 'Phetchabun': 'เพชรบูรณ์', 'Phetchaburi': 'เพชรบุรี',
    'Phichit': 'พิจิตร', 'Phitsanulok': 'พิษณุโลก', 'Phra Nakhon Si Ayutthaya': 'พระนครศรีอยุธยา',
    'Phrae': 'แพร่', 'Phuket': 'ภูเก็ต', 'Prachin Buri': 'ปราจีนบุรี',
    'Prachuap Khiri Khan': 'ประจวบคีรีขันธ์', 'Ranong': 'ระนอง', 'Ratchaburi': 'ราชบุรี',
    'Rayong': 'ระยอง', 'Roi Et': 'ร้อยเอ็ด', 'Sa Kaeo': 'สระแก้ว',
    'Sakon Nakhon': 'สกลนคร', 'Samut Prakan': 'สมุทรปราการ', 'Samut Sakhon': 'สมุทรสาคร',
    'Samut Songkhram': 'สมุทรสงคราม', 'Saraburi': 'สระบุรี', 'Satun': 'สตูล',
    'Sing Buri': 'สิงห์บุรี', 'Songkhla': 'สงขลา', 'Sukhothai': 'สุโขทัย',
    'Suphan Buri': 'สุพรรณบุรี', 'Surat Thani': 'สุราษฎร์ธานี', 'Surin': 'สุรินทร์',
    'Tak': 'ตาก', 'Trang': 'ตรัง', 'Trat': 'ตราด', 'Ubon Ratchathani': 'อุบลราชธานี',
    'Udon Thani': 'อุดรธานี', 'Uttaradit': 'อุตรดิตถ์', 'Yala': 'ยะลา',
    'Chanthaburi': 'จันทบุรี', 'Chaiyaphum': 'ชัยภูมิ', 'Kamphaeng Phet': 'กำแพงเพชร',
    'Kanchanaburi': 'กาญจนบุรี', 'Aranyaprathet': 'สระแก้ว', 'Bhumibol Dam': 'ตาก',
    'Bua Chum': 'ลพบุรี', 'Chok Chai': 'นครราชสีมา', 'Hua Hin': 'ประจวบคีรีขันธ์',
    'Kabin Buri': 'ปราจีนบุรี', 'Khlong Yai': 'ตราด', 'Kosum Phisai': 'มหาสารคาม',
    'Lom Sak': 'เพชรบูรณ์', 'Mae Sariang': 'แม่ฮ่องสอน', 'Mae Sot': 'ตาก',
    'Mukdahan': 'มุกดาหาร', 'Nang Rong': 'บุรีรัมย์', 'Nong Bua Lam Phu': 'หนองบัวลำภู',
    'Phichit Agromet': 'พิจิตร', 'Sawi Agromet': 'ชุมพร', 'Suphan Buri Agromet': 'สุพรรณบุรี',
    'Surin Agromet': 'สุรินทร์', 'Tha Kaho': 'เลย', 'Thong Pha Phum': 'กาญจนบุรี',
    'Wichian Buri': 'เพชรบูรณ์', 'Ko Samui': 'สุราษฎร์ธานี', 'Sadao': 'สงขลา',
    'Takfa Agromet': 'นครสวรรค์', 'Tha Wang Pha': 'น่าน', 'U Thong Agromet': 'สุพรรณบุรี',
    'Doi Mu Soe': 'ตาก', 'Lop Buri': 'ลพบุรี', 'Phatthaya': 'ชลบุรี', 'Laem Chabang': 'ชลบุรี'
}

# สกัดข้อมูลจาก PDF เฉพาะปี 63-64
results_old = []
pdf_files = glob.glob(os.path.join(folder_63_64, '*.pdf'))
print(f"กำลังอ่านข้อมูลในปี 63-64 จำนวน {len(pdf_files)} ไฟล์...")

for file in pdf_files:
    filename = os.path.basename(file)
    # ดึงชื่อเดือนและปีจากชื่อไฟล์
    month_eng = next((en for th, en in thai_months.items() if th in filename), "Unknown")
    year_match = re.search(r'256[3-4]', filename)
    year = "20" + str(int(year_match.group()[-2:]) - 43) if year_match else "Unknown"

    if month_eng == "Unknown" or year == "Unknown": continue

    # อ่านตาราง PDF
    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                for row in table:
                    if len(row) > 2:
                        # คลีนชื่อสถานี ลบช่องว่างออก
                        station = str(row[1]).strip().replace(' ', '')
                        rain = str(row[2]).strip()

                        # วนลูปเทียบชื่อสถานีกับ Dictionary
                        for st_key, prov in thai_station_to_prov.items():
                            if st_key in station:
                                try:
                                    # แปลงค่า '-' และ 'T' ให้เป็น 0.0 (ฝนตกน้อยมากจนวัดไม่ได้)
                                    rain_val = 0.0 if rain in ['-', 'T'] else float(rain)
                                    results_old.append({'Year': int(year), 'Month': month_eng, 'Province': prov, 'Rainfall': rain_val})
                                except: pass
                                break # เจอจังหวัดแล้วหยุดหาต่อ

# นำข้อมูลเข้าสู่ DataFrame และหาค่าเฉลี่ยฝนกรณีที่มีหลายสถานีในจังหวัดเดียว
df_old = pd.DataFrame(results_old)
if not df_old.empty:
    df_old = df_old.groupby(['Year', 'Month', 'Province'], as_index=False)['Rainfall'].mean()

# โหลดข้อมูลปี 65-67 และทำการรวมร่าง
df_new = pd.read_csv(path_cleaned_65_67)
# แปลงชื่อสถานีอังกฤษ ให้เป็นจังหวัดภาษาไทย ด้วยฟังก์ชัน .map()
df_new['Province'] = df_new['Province'].map(eng_station_to_prov).fillna(df_new['Province'])
# ยุบรวมหาค่าเฉลี่ย
df_new = df_new.groupby(['Year', 'Month', 'Province'], as_index=False)['Rainfall'].mean()

# นำข้อมูลปี 63-64 และ 65-67 มาต่อท้ายกัน
df_all = pd.concat([df_old, df_new], ignore_index=True)

# การเติมเต็มข้อมูล
# ปัญหา: หลังจากรวมข้อมูล พบว่าบางจังหวัดไม่มีข้อมูลฝนในบางเดือน
# วิธีแก้: ใช้คำสั่ง transform เติมค่าว่างด้วย "ค่าเฉลี่ยฝนของจังหวัดทั้งหมดในเดือนและปีนั้นๆ"
df_all['Rainfall'] = df_all.groupby(['Year', 'Month'])['Rainfall'].transform(lambda x: x.fillna(x.mean()))

# บันทึกเป็นไฟล์ CSV สุดท้าย
df_all.to_csv('Cleaned_Rainfall_MonthlyV2.csv', index=False, encoding='utf-8-sig')
print(f" จำนวนข้อมูลทั้งหมด {len(df_all)} แถว")
display(df_all.head())

กำลังอ่านข้อมูลในปี 63-64 จำนวน 24 ไฟล์...
 จำนวนข้อมูลทั้งหมด 2029 แถว


,Year,Month,Province,Rainfall
0,2021,Aug,กรุงเทพมหานคร,266.6
1,2021,Aug,ขอนแก่น,190.8
2,2021,Aug,จันทบุรี,623.4
3,2021,Aug,ฉะเชิงเทรา,188.8
4,2021,Aug,ชัยนาท,173.0


In [4]:
import pandas as pd
import numpy as np

# 1. โหลดไฟล์ข้อมูล (ตรวจสอบชื่อไฟล์ให้ตรงกับที่อัปโหลดใน Colab)
df1 = pd.read_csv('Cleaned_Rainfall_Monthly.csv')
df2 = pd.read_csv('Cleaned_Rainfall_MonthlyV2.csv')

# 2. ปรับมาตรฐานชื่อจังหวัด (English -> Thai)
# เพื่อให้ Join กับข้อมูลประชากรและรถยนต์ได้
province_map = {
    'Bangkok Metropolis': 'กรุงเทพมหานคร', 'Bangkok Airport': 'กรุงเทพมหานคร',
    'Aranyaprathet': 'สระแก้ว', 'Chiang Mai': 'เชียงใหม่', 'Chiang Rai': 'เชียงราย',
    'Khon Kaen': 'ขอนแก่น', 'Nan': 'น่าน', 'Pattaya': 'ชลบุรี',
    'Phuket Airport': 'ภูเก็ต', 'Kanchanaburi': 'กาญจนบุรี', 'Kamphaeng Phet': 'กำแพงเพชร',
    'Chon Buri': 'ชลบุรี', 'Chanthaburi': 'จันทบุรี', 'Chumphon': 'ชุมพร',
    'Phrae': 'แพร่', 'Loei': 'เลย', 'Tak': 'ตาก', 'Takua Pa': 'พังงา',
    'Trang Airport': 'ตรัง', 'Pattani Airport': 'ปัตตานี', 'Sattahip': 'ชลบุรี',
    'Umphang': 'ตาก'
}

# รวมร่างข้อมูลและจัดการชื่อจังหวัด
df_combined = pd.concat([df1, df2], ignore_index=True)
df_combined['Province'] = df_combined['Province'].replace(province_map)
df_combined = df_combined.drop_duplicates(subset=['Year', 'Month', 'Province'])

# 3. คำนวณค่าเฉลี่ยรายเดือนของแต่ละจังหวัด (Monthly Mean)
# ใช้สำหรับเติมในเดือนที่ข้อมูลหายไปทั้งแถว
monthly_avg = df_combined.groupby(['Province', 'Month'])['Rainfall'].mean().reset_index()
monthly_avg.rename(columns={'Rainfall': 'Monthly_Mean'}, inplace=True)

# 4. สร้างโครงตารางที่สมบูรณ์ (ทุกจังหวัด ทุกเดือน ทุกปี)
years = df_combined['Year'].unique()
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
provinces = df_combined['Province'].unique()

full_index = pd.MultiIndex.from_product([years, months, provinces], names=['Year', 'Month', 'Province'])
full_df = pd.DataFrame(index=full_index).reset_index()

# 5. กระบวนการ Imputation (การเติมค่าที่หายไป)
# นำข้อมูลจริงมาวางทับโครงตาราง
final_df = pd.merge(full_df, df_combined, on=['Year', 'Month', 'Province'], how='left')
# นำค่าเฉลี่ยรายเดือนมาใส่
final_df = pd.merge(final_df, monthly_avg, on=['Province', 'Month'], how='left')

# ถ้า Rainfall เป็นค่าว่าง (NaN) ให้แทนที่ด้วย Monthly_Mean
final_df['Rainfall_Final'] = final_df['Rainfall'].fillna(final_df['Monthly_Mean'])

# 6. บันทึกผลลัพธ์
output_filename = 'Rainfall_Monthly_Imputed.csv'
final_df[['Year', 'Month', 'Province', 'Rainfall_Final']].to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"จำนวนแถวทั้งหมด {len(final_df)} แถว")
display(final_df.head(10))

จำนวนแถวทั้งหมด 4848 แถว


,Year,Month,Province,Rainfall,Monthly_Mean,Rainfall_Final
0,2022,Jan,สระแก้ว,9.5,5.000000,9.5
1,2022,Jan,กรุงเทพมหานคร,29.1,9.766667,29.1
2,2022,Jan,Bhumibol Dam,2.2,6.533333,2.2
3,2022,Jan,Bua Chum,65.4,21.800000,65.4
4,2022,Jan,Chaiyaphum,24.1,8.666667,24.1
5,2022,Jan,จันทบุรี,19.7,7.666667,19.7
6,2022,Jan,เชียงใหม่,26.7,10.366667,26.7
7,2022,Jan,เชียงราย,89.0,32.600000,89.0
8,2022,Jan,Chok Chai,0.1,5.500000,0.1
9,2022,Jan,ชลบุรี,1.8,0.600000,1.8


In [14]:
import pandas as pd
import numpy as np
import os # Import the os module to check for file existence

# 1. โหลดข้อมูล
# แก้ไขชื่อไฟล์ที่โหลดให้เป็น 'Rainfall_Monthly_Imputed.csv' ที่สร้างจากขั้นตอนก่อนหน้า
file_to_load = 'Rainfall_Monthly_Imputed.csv'

try:
    df = pd.read_csv(file_to_load)
except FileNotFoundError:
    print(f"Error: The file '{file_to_load}' was not found.")
    print("Please ensure you have run the previous cell (JGuW2bYN_YEz) to generate this file.")
    # Exit or set an empty DataFrame to prevent further errors if file is critical
    df = pd.DataFrame() # Create an empty DataFrame to avoid errors in subsequent steps
    # You might want to add a 'return' here or raise an exception if an empty DF is not acceptable

if not df.empty:
    # 2. แก้ไขชื่อจังหวัดให้เป็นมาตรฐาน (ไทย)
    # (สร้าง Dictionary สำหรับ Map ชื่อที่พบบ่อยในไฟล์)
    mapping = {
        'Surat Thani': 'สุราษฎร์ธานี', 'Udon Thani': 'อุดรธานี', 'Nan': 'น่าน',
        'Trang': 'ตรัง', 'Tak': 'ตาก', 'Phuket': 'ภูเก็ต', 'Chiang Mai': 'เชียงใหม่',
        'Bhumibol Dam': 'ตาก', 'Bua Chum': 'ลพบุรี', 'Chaiyaphum': 'ชัยภูมิ'
    }
    df['Province'] = df['Province'].replace(mapping)

    # 3. จัดการค่าผิดปกติ (ติดลบ) และค่าว่าง
    # เปลี่ยนค่าติดลบให้เป็น 0
    # เปลี่ยนจาก 'Rainfall' เป็น 'Rainfall_Final'
    df.loc[df['Rainfall_Final'] < 0, 'Rainfall_Final'] = 0

    # 4. ยุบรวมข้อมูลที่ซ้ำกันจากการเปลี่ยนชื่อ (Group by และหาค่าเฉลี่ย)
    # เปลี่ยนจาก 'Rainfall' เป็น 'Rainfall_Final'
    df_clean = df.groupby(['Year', 'Month', 'Province'])['Rainfall_Final'].mean().reset_index()

    # 5. เติมค่าว่าง (Imputation) ด้วยค่าเฉลี่ยรายเดือนของจังหวัดนั้นๆ
    # หาค่าเฉลี่ยรายเดือนอ้างอิง
    # เปลี่ยนจาก 'Rainfall' เป็น 'Rainfall_Final'
    monthly_avg = df_clean.groupby(['Province', 'Month'])['Rainfall_Final'].transform('mean')
    df_clean['Rainfall_Final'] = df_clean['Rainfall_Final'].fillna(monthly_avg)

    # กรณีที่ยังว่างอยู่ (เช่น จังหวัดนั้นไม่มีข้อมูลเดือนนั้นเลยทุกปี) ให้เติมด้วย 0
    # เปลี่ยนจาก 'Rainfall' เป็น 'Rainfall_Final'
    df_clean['Rainfall_Final'] = df_clean['Rainfall'].fillna(0)

    # 6. บันทึกไฟล์ใหม่
    df_clean.to_csv('Final_Rainfall_Ready.csv', index=False, encoding='utf-8-sig')

    print(f" ข้อมูลคงเหลือ: {len(df_clean)} แถว")
display(df_clean.head(10))


 ข้อมูลคงเหลือ: 4560 แถว


,Year,Month,Province,Rainfall_Final
0,2021,Apr,Chok Chai,66.383333
1,2021,Apr,Hua Hin,11.366667
2,2021,Apr,Kabin Buri,45.000000
3,2021,Apr,Khlong Yai,102.233333
4,2021,Apr,Kosum Phisai,45.900000
5,2021,Apr,Lampang,38.800000
6,2021,Apr,Lamphun,21.200000
7,2021,Apr,Lom Sak,45.066667
8,2021,Apr,Lop Buri,34.866667
9,2021,Apr,Mae Sariang,58.166667
